# Q2 · Evolution of fire causes
One overlaid line chart compares six cause groups on the same annual percentage scale. This makes rankings and crossings easier to compare than separate panels; individual trends can be harder to follow where lines overlap. Colour-blind-friendly colours and different dash patterns identify causes redundantly. Hover for counts and annual totals.

**Manual grouping, not clustering:** 210–219 = agricultural burning; 100 = lightning; 400 = intentional; 500 (or missing code) = unknown; 600 = rekindling. All remaining codes = other recorded causes. Source files are unchanged. This reduces dozens of detailed causes to six readable lines, but “Other” hides its subcategories.

**Coverage:** GovEsp, 1998–2022. GenCat-only rows have no causes, so 2023–2024 cannot answer Q2. Percentages include unknown causes and separate composition from overall fire counts; a rising share does not necessarily mean a rising count.

The design follows the notes: time on x, position on a common scale, categorical colour with redundant dashes, and no stacked baselines (pp. 29, 60, 66–67). No downloads. Run all cells from the project folder.


In [1]:
import pandas as pd
import altair as alt
from IPython.display import display
fires = pd.read_csv("Clean data/CleanData.csv", parse_dates=["data_incendi"])
fires = fires[fires.font.eq("GovEsp") & fires.data_incendi.dt.year.between(1998, 2022)].copy()
fires["year"] = fires.data_incendi.dt.year
fires["cause"] = fires.codi_causa.map({100: "Lightning", 400: "Intentional", 500: "Unknown", 600: "Rekindling"}).fillna("Other recorded causes")
fires.loc[fires.codi_causa.between(210, 219), "cause"] = "Agricultural burning"
fires.loc[fires.codi_causa.isna(), "cause"] = "Unknown"
groups = ["Intentional", "Agricultural burning", "Other recorded causes", "Lightning", "Rekindling", "Unknown"]
annual = fires.groupby(["year", "cause"]).size().reindex(pd.MultiIndex.from_product([range(1998, 2023), groups], names=["year", "cause"]), fill_value=0).reset_index(name="fires")
annual["total"] = annual.groupby("year").fires.transform("sum")
annual["share"] = annual.fires / annual.total
print(f"{len(fires):,} GovEsp fires; missing GenCat causes are not inferred.")

15,366 GovEsp fires; missing GenCat causes are not inferred.


In [2]:
chart = alt.Chart(annual).mark_line(strokeWidth=2.5).encode(
    x=alt.X("year:Q", title="Year", scale=alt.Scale(zero=False), axis=alt.Axis(format="d", tickCount=8)),
    y=alt.Y("share:Q", title="Share of annual fires", axis=alt.Axis(format=".0%"), scale=alt.Scale(zero=True)),
    color=alt.Color("cause:N", title="Cause group", scale=alt.Scale(domain=groups, range=["#0072b2", "#e69f00", "#009e73", "#cc79a7", "#56b4e9", "#666666"])),
    strokeDash=alt.StrokeDash("cause:N", title="Cause group", sort=groups),
    tooltip=["cause:N", alt.Tooltip("year:O", title="Year"), alt.Tooltip("share:Q", format=".1%"), "fires:Q", alt.Tooltip("total:Q", title="All fires that year")]
).properties(width=720, height=370, title="Q2 · How the cause mix changed (GovEsp, 1998–2022)")
spec = chart.to_dict()
spec["$schema"] = "https://vega.github.io/schema/vega-lite/v5.json"  # VS Code compatibility
display({"application/vnd.vegalite.v5+json": spec, "text/html": chart.to_html()}, raw=True)


<!DOCTYPE html>

## Alternative · Seven meaningful cause groups, directly labelled
This is **manual grouping by the recorded cause**, not statistical clustering. Each fire belongs to one group:

| Group | Rule / examples |
|---|---|
| Intentional | 400; also 335, explicitly intentional vehicle burning |
| Unknown | 500 or missing cause code |
| Lightning | 100 |
| Vegetation / waste burning | 210–239, 260–289, 295, 370: agricultural, livestock, forestry, vegetation clearance, waste/landfill and escaped controlled burns |
| Equipment / infrastructure | 310–369, except 335: railways, electricity, machinery, vehicles, military activities and tools |
| Rekindling | 600 |
| Other recorded causes | Remaining codes: smoking, campfires/barbecues, fireworks, miscellaneous activities and unspecified recorded causes |

The original labels remain unchanged. Explicit intentional vehicle burning takes priority over equipment; unknown is not mixed with other recorded causes.

**Solid lines with names at their ends** remove the need to decode dash patterns or a separate legend. Short connectors separate nearby endpoint labels without moving data points. Colours use an accessible categorical palette; hover confirms group, percentage and count.

The y-axis remains percentage of all GovEsp fires that year, including unknown causes. Coverage is 1998–2022; missing GenCat causes cannot be inferred. Grouping improves readability but hides variation within groups.


In [3]:
grouped = fires.copy()
c = grouped.codi_causa
grouped["group"] = "Other recorded causes"
grouped.loc[c.between(210, 239) | c.between(260, 289) | c.isin([295, 370]), "group"] = "Vegetation / waste burning"
grouped.loc[c.between(310, 369), "group"] = "Equipment / infrastructure"
for codes, name in [([100], "Lightning"), ([335, 400], "Intentional"), ([500], "Unknown"), ([600], "Rekindling")]: grouped.loc[c.isin(codes), "group"] = name
grouped.loc[c.isna(), "group"] = "Unknown"
cause_groups = ["Intentional", "Unknown", "Lightning", "Vegetation / waste burning", "Equipment / infrastructure", "Other recorded causes", "Rekindling"]
trends = grouped.groupby(["year", "group"]).size().reindex(pd.MultiIndex.from_product([range(1998, 2023), cause_groups], names=["year", "group"]), fill_value=0).reset_index(name="fires")
trends["share"] = trends.fires / trends.groupby("year").fires.transform("sum")
ends = trends[trends.year.eq(2022)].sort_values("share").copy()
positions = []
for value in ends.share: positions.append(max(value, positions[-1] + 0.025) if positions else value)
ends["label_share"] = positions


In [4]:
base = alt.Chart(trends).encode(
    x=alt.X("year:Q", title="Year", scale=alt.Scale(domain=[1998, 2023]), axis=alt.Axis(format="d", values=[1998, 2002, 2006, 2010, 2014, 2018, 2022])),
    y=alt.Y("share:Q", title="Percentage of annual fires", axis=alt.Axis(format=".0%"), scale=alt.Scale(zero=True)),
    color=alt.Color("group:N", legend=None, scale=alt.Scale(domain=cause_groups, range=["#d55e00", "#777777", "#0072b2", "#009e73", "#cc79a7", "#e69f00", "#000000"])),
    tooltip=[alt.Tooltip("group:N", title="Cause group"), "year:O", "fires:Q", alt.Tooltip("share:Q", title="Percentage", format=".1%")])
lines = base.mark_line(strokeWidth=2.5)
connectors = base.mark_rule(strokeWidth=1).encode(x2=alt.datum(2023), y2="label_share:Q").properties(data=ends)
labels = base.mark_text(align="left", dx=6, fontSize=12).encode(x=alt.datum(2023), y=alt.Y("label_share:Q", title="Percentage of annual fires"), text="group:N").properties(data=ends)
group_chart = (lines + connectors + labels).properties(width=700, height=420, title="Q2 · Cause groups over time (GovEsp, 1998–2022)")
spec = group_chart.to_dict()
spec["$schema"] = "https://vega.github.io/schema/vega-lite/v5.json"
display({"application/vnd.vegalite.v5+json": spec, "text/html": group_chart.to_html()}, raw=True)


<!DOCTYPE html>

## Three-year moving average · Underlying cause trends
For each group and year **t**, plot the mean of its annual percentages in **t−1, t and t+1**. For example, the point at 2010 averages 2009–2011. Each year has equal weight; this is not the percentage calculated from pooled three-year fire counts.

**Why three years and centered?** Three years reduce isolated annual fluctuations while preserving more detail than a five-year window. Centering avoids shifting the historical trend to a later year, as a trailing average would. This is a descriptive smoothing choice, not an estimated optimal window or a forecast: it uses the following year’s data.

Only complete windows are shown: **1999–2021**, calculated from 1998–2022. Endpoints are omitted rather than using shorter windows or inventing observations. The same seven groups, colours and direct labels are retained. Percentages still sum to 100% at each displayed year. Compare with the unchanged annual chart above: smoothing intentionally hides short-lived peaks and does not remove source or classification limitations.


In [5]:
smoothed = trends.sort_values(["group", "year"]).copy()
smoothed["share"] = smoothed.groupby("group").share.transform(lambda s: s.rolling(3, center=True, min_periods=3).mean())
smoothed = smoothed.dropna(subset=["share"])
smoothed["start"], smoothed["end"] = smoothed.year - 1, smoothed.year + 1
smooth_ends = smoothed[smoothed.year.eq(2021)].sort_values("share").copy()
label_positions = []
for value in smooth_ends.share: label_positions.append(max(value, label_positions[-1] + 0.025) if label_positions else value)
smooth_ends["label_share"] = label_positions


In [6]:
smooth_base = base.properties(data=smoothed).encode(
    y=alt.Y("share:Q", title="3-year mean of annual percentages", axis=alt.Axis(format=".0%"), scale=alt.Scale(domain=[0, max(trends.share.max(), ends.label_share.max())], nice=True)),
    tooltip=[alt.Tooltip("group:N", title="Cause group"), alt.Tooltip("year:O", title="Center year"), alt.Tooltip("start:O", title="Window starts"), alt.Tooltip("end:O", title="Window ends"), alt.Tooltip("share:Q", title="3-year mean", format=".1%")])
smooth_lines = smooth_base.mark_line(strokeWidth=2.5)
smooth_links = smooth_base.mark_rule(strokeWidth=1).encode(x2=alt.datum(2022), y2="label_share:Q").properties(data=smooth_ends)
smooth_labels = smooth_base.mark_text(align="left", dx=6, fontSize=12).encode(x=alt.datum(2022), y=alt.Y("label_share:Q", title="3-year mean of annual percentages"), text="group:N").properties(data=smooth_ends)
moving_chart = (smooth_lines + smooth_links + smooth_labels).properties(width=700, height=420, title="Q2 · Centered 3-year moving average (1999–2021)")
spec = moving_chart.to_dict()
spec["$schema"] = "https://vega.github.io/schema/vega-lite/v5.json"
display({"application/vnd.vegalite.v5+json": spec, "text/html": moving_chart.to_html()}, raw=True)


<!DOCTYPE html>